# 01 - Import and version an existing skill

Import an existing SKILL.md into Foundry and manage its versions. The included engineering-impact-assessment skill is a sample; point SKILL_DIR at your own single-file skill to replace it.

Register the skill using the Python SDK, create an updated version, then select the default. Notebook 02 attaches it to Toolbox for later consumption from MAF.

**Prerequisites:** follow the [Azure setup](../README.md#azure-setup), copy ../.env.example to ../.env, install ../requirements.txt and select that Python kernel. Authenticate with `az login --tenant <your-tenant-id>`. These cells make real service calls; each create call stores a new version.

## Configure the project

The cell loads the workspace-root .env. Set FOUNDRY_PROJECT_ENDPOINT to the project endpoint, not a model endpoint. Optionally set FOUNDRY_SKILL_DIR to your single-file skill folder; its YAML name becomes the Foundry asset name. Relative directories are resolved from the workspace root. Existing process environment variables take precedence; restart the kernel after changing .env. Skills are a preview API.

In [ ]:
import os
import re
from pathlib import Path
from urllib.parse import urlsplit

import yaml
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import SkillInlineContent
from azure.identity import AzureCliCredential

ROOT = Path.cwd().resolve()
if not (ROOT / "requirements.txt").is_file():
    ROOT = ROOT.parent
if not (ROOT / "requirements.txt").is_file() or not (ROOT / ".env.example").is_file():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")
load_dotenv(ROOT / ".env", override=False, interpolate=False)

# Azure CLI accepts a subscription or tenant selector, not both.
subscription = os.environ.get("AZURE_SUBSCRIPTION_ID", "").strip()
tenant = os.environ.get("AZURE_TENANT_ID", "").strip()
CREDENTIAL_OPTIONS = {"subscription": subscription} if subscription else ({"tenant_id": tenant} if tenant else {})
PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "").rstrip("/")
endpoint = urlsplit(PROJECT_ENDPOINT)
if (
    endpoint.scheme != "https" or not endpoint.hostname
    or not endpoint.hostname.endswith(".services.ai.azure.com")
    or endpoint.username is not None or endpoint.password is not None
    or endpoint.port not in (None, 443) or endpoint.query or endpoint.fragment
    or not re.fullmatch(r"/api/projects/[^/]+", endpoint.path)
):
    raise ValueError("Set FOUNDRY_PROJECT_ENDPOINT to https://<resource>.services.ai.azure.com/api/projects/<project>.")

skill_directory = os.environ.get("FOUNDRY_SKILL_DIR", "").strip()
SKILL_DIR = Path(skill_directory) if skill_directory else ROOT / ".github" / "skills" / "engineering-impact-assessment"
if not SKILL_DIR.is_absolute():
    SKILL_DIR = ROOT / SKILL_DIR
print("Project:", PROJECT_ENDPOINT)

## Read the skill

SKILL.md contains discovery metadata (`name`, `description`) and the procedure an agent loads when relevant. This import uses inline content for a single-file skill. Skills with supporting files require a package upload, not this inline path.

In [ ]:
skill_text = (SKILL_DIR / "SKILL.md").read_bytes().decode("utf-8")
lines = skill_text.splitlines()
if not lines or lines[0] != "---":
    raise ValueError("SKILL.md must start with YAML front matter.")
try:
    front_matter_end = lines.index("---", 1)
except ValueError as error:
    raise ValueError("SKILL.md has no closing front matter delimiter.") from error
metadata = yaml.safe_load("\n".join(lines[1:front_matter_end]))
if not isinstance(metadata, dict):
    raise ValueError("Skill metadata must be a YAML mapping.")
name = metadata.get("name")
description = metadata.get("description")
if not isinstance(name, str) or not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", name):
    raise ValueError("Skill name must be lowercase words separated by hyphens.")
if len(name) > 64 or name != SKILL_DIR.name:
    raise ValueError("Skill name must match the directory and be at most 64 characters.")
if not isinstance(description, str) or not 1 <= len(description) <= 1024:
    raise ValueError("Skill description must contain 1-1024 characters.")
# Inline upload takes the Markdown body, without YAML front matter.
instructions = "\n".join(lines[front_matter_end + 1:]).strip()
if not instructions:
    raise ValueError("Skill instructions must not be empty.")
if any(path.is_file() and path != SKILL_DIR / "SKILL.md" for path in SKILL_DIR.rglob("*")):
    raise ValueError("This inline import supports only SKILL.md. Use a package upload for skills with supporting files.")
SKILL_NAME = name
content = SkillInlineContent(description=description, instructions=instructions)
print(skill_text)

## Register a skill version

`skills.create` stores an immutable snapshot under SKILL_NAME. If that name already exists, it adds a version. Record the returned identifier rather than assuming a version number.

`default=False` avoids requesting default promotion during upload. Inspect the returned asset metadata for the service's first-version default behavior.

In [ ]:
with AzureCliCredential(**CREDENTIAL_OPTIONS) as credential, AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
    initial_version = project.beta.skills.create(name=SKILL_NAME, inline_content=content, default=False)
    current = project.beta.skills.get(SKILL_NAME)
    print("Created version:", initial_version.version)
    print("Skill metadata:", current.as_dict())

## Create an updated version

Add rollout and rollback checks to the procedure. The update is built in memory; SKILL.md and the previous stored version remain unchanged. Rerunning this cell creates another version.

In [ ]:
updated_instructions = instructions + "\n\n## Release-impact checklist\nIdentify rollout, monitoring and rollback decisions requiring owner approval.\n"
updated_content = SkillInlineContent(description=metadata["description"], instructions=updated_instructions)
with AzureCliCredential(**CREDENTIAL_OPTIONS) as credential, AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
    updated_version = project.beta.skills.create(name=SKILL_NAME, inline_content=updated_content, default=False)
    print("Updated version:", updated_version.version)

## Publish the selected version

**Publish/promote** means select a reviewed version as the default. It does not make the skill public, grant permissions or deploy an agent. Inspect the updated instructions above before running this cell.

A consumer pinned to a specific version remains unchanged. Default-following consumers may cache content. To roll back, use an older retained identifier as SELECTED_VERSION; this does not undo prior agent actions.

Copy the printed FOUNDRY_SKILL_NAME and FOUNDRY_SKILL_VERSION into .env before running notebook 02.

In [ ]:
SELECTED_VERSION = updated_version.version
with AzureCliCredential(**CREDENTIAL_OPTIONS) as credential, AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as project:
    project.beta.skills.get_version(SKILL_NAME, SELECTED_VERSION)
    # Change the default pointer; existing versions are not modified.
    published = project.beta.skills.update(name=SKILL_NAME, default_version=SELECTED_VERSION)
    assert published.default_version == SELECTED_VERSION
    print("Default version:", published.default_version)
    print(f"FOUNDRY_SKILL_NAME={SKILL_NAME}")
    print(f"FOUNDRY_SKILL_VERSION={SELECTED_VERSION}")

## Verify in the Foundry portal

After running the publish cell above, open [Microsoft Foundry](https://ai.azure.com) and select the same project as PROJECT_ENDPOINT. Go to **Tools > Skills (Preview)**. Find the skill named SKILL_NAME and compare its displayed version with SELECTED_VERSION from the cell output.

The example below shows `engineering-impact-assessment` at version **2**. Your version may differ if you reran the create cells.

![Foundry Tools > Skills showing engineering-impact-assessment version 2](images/foundry-skills.png)

Open the skill, select the version you published using the version selector at the top, then choose **Code** to inspect SKILL.md. You can review the metadata and instructions imported by the SDK. Scroll down to confirm the **Release-impact checklist** added in the update cell.

![Foundry skill detail showing the version selector and SKILL.md instructions in Code view](images/foundry-skill-content.png)